# **Machine Learning on Big Data (CN7030) CRWK 23-24 Term B [60% weighting]**
# **Group ID: [35]**
1.   Student 1: Morayo Ogun u1313791

---

If you want to add comments on your group work, please write it here for us:


# **Initiate and Configure Spark**

In [ ]:
!pip3 install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488491 sha256=e8eab9a76b0baccb88c441f4cc87d63735140ef4db57539528a654747ca05da5
  Stored in directory: /root/.cache/pip/wheels/80/1d/60/2c256ed38dddce2fdd93be545214a63e02fbd8d74fb0b7f3a6
Successfully built pyspark


In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
                    .appName("LogisticRegression") \
                    .master("local[*]") \
                    .config("spark.executor.memory", "4g") \
                    .config("spark.driver.memory", "2g") \
                    .config("spark.executor.cores", "2") \
                    .config("spark.sql.inMemoryColumnarStorage.compressed", "true") \
                    .getOrCreate()

spark

---
# **Task 1 - Data Loading and Preprocessing (15 marks)**
---

In [ ]:
# Student number 1313791, Morayo Ogun

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = spark.read.csv("/content/drive/MyDrive/Files/UniversityServerLogsDataset.csv", header = True, inferSchema=True)

In [ ]:
df.show(5)
# more info
print(df.count())
print(df.rdd.getNumPartitions())

+--------+--------+-------------------+-------------+------------+------------+---------------+---------------+---------------+---------------+----------------+---------------+---------------+---------------+----------------+---------------+--------------+------------+----------------+----------------+------------+------------+-----------+----------------+----------------+-----------+-----------+-----------+----------------+----------------+-----------+-----------+-------------+-------------+-------------+-------------+--------------+--------------+------------+------------+-----------+-----------+--------------+--------------+----------------+------------+------------+------------+------------+------------+------------+--------------+------------+-------------+------------+----------------+----------------+--------------+--------------+----------------+--------------+--------------+----------------+----------------+----------------+----------------+----------------+-----------------+-

**Check for NAN or NULL values in the dataset**

In [ ]:
from pyspark.sql.functions import col, isnan, when, count

# Count NaN and Null values in each column
nan_counts = df.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in df.columns])

# Show the counts
nan_counts.show()

+--------+--------+---------+-------------+------------+------------+---------------+---------------+---------------+---------------+----------------+---------------+---------------+---------------+----------------+---------------+-----------+-----------+-------------+------------+------------+------------+-----------+------------+-----------+-----------+-----------+-----------+------------+-----------+-----------+-----------+-------------+-------------+-------------+-------------+--------------+--------------+----------+----------+-----------+-----------+------------+-----------+-----------+------------+------------+------------+------------+------------+------------+--------------+------------+-------------+------------+----------------+----------------+--------------+--------------+----------------+--------------+--------------+----------------+----------------+----------------+----------------+----------------+-----------------+-----------------+-----------------+----------------+-

**Drop a column that had some NAN or NULL values**

In [ ]:
df = df.drop("Flow Byts/s")
df.show(5)

+--------+--------+-------------------+-------------+------------+------------+---------------+---------------+---------------+---------------+----------------+---------------+---------------+---------------+----------------+---------------+------------+----------------+----------------+------------+------------+-----------+----------------+----------------+-----------+-----------+-----------+----------------+----------------+-----------+-----------+-------------+-------------+-------------+-------------+--------------+--------------+------------+------------+-----------+-----------+--------------+--------------+----------------+------------+------------+------------+------------+------------+------------+--------------+------------+-------------+------------+----------------+----------------+--------------+--------------+----------------+--------------+--------------+----------------+----------------+----------------+----------------+----------------+-----------------+----------------

**Check for inifnity values in a dataset**

In [ ]:
from pyspark.sql.functions import col, when, count

columns_to_check = ['']
# Count infinity values in each column
inf_counts = df.select([count(when((col(c) == float('inf')) | (col(c) == float('-inf')), c)).alias(c) for c in columns_to_check])

# Show the counts
inf_counts.show()

+--------+--------+---------+-------------+------------+------------+---------------+---------------+---------------+---------------+----------------+---------------+---------------+---------------+----------------+---------------+-----------+-------------+------------+------------+------------+-----------+------------+-----------+-----------+-----------+-----------+------------+-----------+-----------+-----------+-------------+-------------+-------------+-------------+--------------+--------------+----------+----------+-----------+-----------+------------+-----------+-----------+------------+------------+------------+------------+------------+------------+--------------+------------+-------------+------------+----------------+----------------+--------------+--------------+----------------+--------------+--------------+----------------+----------------+----------------+----------------+----------------+-----------------+-----------------+-----------------+----------------+-----------+-

**Drop a column that had infinity values**

In [ ]:
df = df.drop("Flow Pkts/s")
df.show(5)

+--------+--------+-------------------+-------------+------------+------------+---------------+---------------+---------------+---------------+----------------+---------------+---------------+---------------+----------------+---------------+----------------+----------------+------------+------------+-----------+----------------+----------------+-----------+-----------+-----------+----------------+----------------+-----------+-----------+-------------+-------------+-------------+-------------+--------------+--------------+------------+------------+-----------+-----------+--------------+--------------+----------------+------------+------------+------------+------------+------------+------------+--------------+------------+-------------+------------+----------------+----------------+--------------+--------------+----------------+--------------+--------------+----------------+----------------+----------------+----------------+----------------+-----------------+-----------------+-----------

**Check the Binary labels**

In [ ]:
# We are showing the unique names in the label column

df.select("Label").distinct().show()

+--------------+
|         Label|
+--------------+
|SSH-Bruteforce|
|        Benign|
|FTP-BruteForce|
+--------------+



In [ ]:
#checking thre are no strings left, we use Label column to compare the prediction to the Label column further down in the code once model implememted
df.printSchema()

root
 |-- Dst Port: integer (nullable = true)
 |-- Protocol: integer (nullable = true)
 |-- Timestamp: string (nullable = true)
 |-- Flow Duration: long (nullable = true)
 |-- Tot Fwd Pkts: integer (nullable = true)
 |-- Tot Bwd Pkts: integer (nullable = true)
 |-- TotLen Fwd Pkts: integer (nullable = true)
 |-- TotLen Bwd Pkts: integer (nullable = true)
 |-- Fwd Pkt Len Max: integer (nullable = true)
 |-- Fwd Pkt Len Min: integer (nullable = true)
 |-- Fwd Pkt Len Mean: double (nullable = true)
 |-- Fwd Pkt Len Std: double (nullable = true)
 |-- Bwd Pkt Len Max: integer (nullable = true)
 |-- Bwd Pkt Len Min: integer (nullable = true)
 |-- Bwd Pkt Len Mean: double (nullable = true)
 |-- Bwd Pkt Len Std: double (nullable = true)
 |-- Flow IAT Mean: double (nullable = true)
 |-- Flow IAT Std: double (nullable = true)
 |-- Flow IAT Max: long (nullable = true)
 |-- Flow IAT Min: long (nullable = true)
 |-- Fwd IAT Tot: long (nullable = true)
 |-- Fwd IAT Mean: double (nullable = true)
 |-

In [ ]:
# Label count

df.groupBy("Label").count().orderBy('count', ascending = False).show()

+--------------+------+
|         Label| count|
+--------------+------+
|        Benign|667626|
|FTP-BruteForce|193360|
|SSH-Bruteforce|187589|
+--------------+------+



In [ ]:
from pyspark.sql.functions import when

df = df.withColumn("label", when(df["Label"] != 'Benign', 1).otherwise(0))

df.select("label").distinct().show()

+-----+
|label|
+-----+
|    1|
|    0|
+-----+



In [ ]:
df = df.select("Dst Port", "Protocol", "Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts", "Label")
df.show(25)

+--------+--------+-------------+------------+------------+-----+
|Dst Port|Protocol|Flow Duration|Tot Fwd Pkts|Tot Bwd Pkts|Label|
+--------+--------+-------------+------------+------------+-----+
|       0|       0|    112641719|           3|           0|    0|
|       0|       0|    112641466|           3|           0|    0|
|       0|       0|    112638623|           3|           0|    0|
|      22|       6|      6453966|          15|          10|    0|
|      22|       6|      8804066|          14|          11|    0|
|      22|       6|      6989341|          16|          12|    0|
|       0|       0|    112640480|           3|           0|    0|
|       0|       0|    112641244|           3|           0|    0|
|      80|       6|       476513|           5|           3|    0|
|      80|       6|       475048|           5|           3|    0|
|      80|       6|       474926|           5|           3|    0|
|      80|       6|       477471|           5|           3|    0|
|      80|

In [ ]:
df.groupBy("Label").count().show()

+-----+------+
|Label| count|
+-----+------+
|    1|380949|
|    0|667626|
+-----+------+



**To assess the linear seperability of a feature**

In [ ]:
# We will calculate the average and standard deviation for Flow Duration by class
from pyspark.sql.functions import avg, stddev

# avg & std for class 0
class_0_stats = df.filter(df['label'] == 0).select(avg('Flow Duration',).alias("avg_Flow Duration_0"), stddev('Flow Duration').alias("stddev_Flow Duration_0")).first()

# avg & std for class 1
class_1_stats = df.filter(df['label'] == 1).select(avg('Flow Duration').alias("avg_Flow Duration_1"),
                                                   stddev('Flow Duration').alias("stddev_Flow Duration_1")).first()


print("Class 0 distribution: ")
print("AVG Flow Duration: ", class_0_stats["avg_Flow Duration_0"])
print("STD Flow Duration: ", class_0_stats["stddev_Flow Duration_0"])

print("Class 1 distribution: ")
print("AVG Flow Duration: ", class_1_stats["avg_Flow Duration_1"])
print("STD Flow Duration: ", class_1_stats["stddev_Flow Duration_1"])

Class 0 distribution: 
AVG Flow Duration:  9773470.558245186
STD Flow Duration:  1579432800.3748097
Class 1 distribution: 
AVG Flow Duration:  90287.98862314902
STD Flow Duration:  158737.19288895125


**StringIndexer**

In [ ]:
# Label is the only string so we convert just Label column from string to numerical
# indexer = StringIndexer(inputCol = 'Label', outputCol = 'Label_indexed')
# df = indexer.fit(df).transform(df)
# df.show(10)

In [ ]:
from pyspark.ml.feature import VectorAssembler

# I chose the columns that are most important for the vector assembler and I am ignoring the rest of the columns.
# Preserve and include invalid records in the output dataset.
assembler = VectorAssembler(inputCols = ["Dst Port", "Protocol", "Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts"],

                            outputCol = "features", handleInvalid = "keep")

data = assembler.transform(df)

data = data.select('features', 'label')
data.show(5)

+--------------------+-----+
|            features|label|
+--------------------+-----+
|(5,[2,3],[1.12641...|    0|
|(5,[2,3],[1.12641...|    0|
|(5,[2,3],[1.12638...|    0|
|[22.0,6.0,6453966...|    0|
|[22.0,6.0,8804066...|    0|
+--------------------+-----+
only showing top 5 rows



In [ ]:
# Convert sparse vector to dense list.

selected_data = data.select('features').limit(2).collect()

for row in selected_data:
  dense_vector = row[0].toArray()
  print(dense_vector)

[0.00000000e+00 0.00000000e+00 1.12641719e+08 3.00000000e+00
 0.00000000e+00]
[0.00000000e+00 0.00000000e+00 1.12641466e+08 3.00000000e+00
 0.00000000e+00]


**Normalization and Standardization**

In [ ]:
from pyspark.ml.feature import StandardScaler

# Apply StandardScaler algorithm to the 'features' column of the data, transforming it into a new column called'scaledFeatures', and then display the first 3 rows of the data

scaler = StandardScaler(inputCol = 'features', outputCol = 'scaledFeatures')

scaler_model = scaler.fit(data)
data = scaler_model.transform(data)

data = data.select("scaledFeatures", "label")
data.show(3, truncate = False)

+---------------------------------------------------+-----+
|scaledFeatures                                     |label|
+---------------------------------------------------+-----+
|(5,[2,3],[0.08937754002104278,0.06744829905434696])|0    |
|(5,[2,3],[0.08937733927377235,0.06744829905434696])|0    |
|(5,[2,3],[0.08937508344574935,0.06744829905434696])|0    |
+---------------------------------------------------+-----+
only showing top 3 rows



In [ ]:
train_data, test_data = data.randomSplit([0.7, 0.3], seed = 1234)

---
# **Task 2 - Model Selection and Implementation (25 marks)**
---


In [ ]:
# Implement the model
lr = LogisticRegression(featuresCol = "scaledFeatures", labelCol = 'label',
                        threshold = 0.5, regParam = 0.01)

lr_model = lr.fit(train_data)

lr_predictions_train = lr_model.transform(train_data)
lr_predictions_test = lr_model.transform(test_data)

In [ ]:
# Displaying predicted labels and actual labels for test data.
lr_predictions_test.select("label", "prediction").show(20)

+-----+----------+
|label|prediction|
+-----+----------+
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
|    0|       1.0|
+-----+----------+
only showing top 20 rows



---
# **Task 3 - Model Parameter Tuning (20 marks)**
---


In [ ]:
from pyspark.ml.classification import LogisticRegression

# A technique to optimize model hyperparameters through cross-validation and evaluation.

from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder,CrossValidator

# Initialize Logistic Regression model
lr = LogisticRegression(featuresCol='scaledFeatures', labelCol='label')

# Define a grid of hyperparameters to search over
param_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 1.0,]) \
    .build()

# Define an evaluator for binary classification
evaluator = BinaryClassificationEvaluator(metricName='areaUnderPR')

# Create a CrossValidator to tune the model
crossval = CrossValidator(estimator=lr,
                          estimatorParamMaps=param_grid,
                          evaluator=evaluator,
                          numFolds=5)

# Fit the CrossValidator to find the best model
cv_model = crossval.fit(train_data)

# Get the best Logistic Regression model from cross-validation
best_lr_model = cv_model.bestModel

# Make predictions on test data using the best model
predictions = best_lr_model.transform(test_data)

# Evaluate the model using the evaluator
auc = evaluator.evaluate(predictions)
print("Area Under ROC (AUC):", auc)

Area Under ROC (AUC): 0.5388483304054053


In [ ]:
# Implement the model with updated parameters
lr = LogisticRegression(featuresCol = "scaledFeatures", labelCol = 'label',
                        threshold = 0.5, regParam = 0.01)

lr_model = lr.fit(train_data)

lr_predictions_train = lr_model.transform(train_data)
lr_predictions_test = lr_model.transform(test_data)

---
# **Task 4 - Model Evaluation and Accuracy Calculation (20 marks)**
---

In [ ]:
confusion_matrix = lr_predictions_test.groupBy("label", "prediction").count()
confusion_matrix.show()

+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|    1|       0.0| 28055|
|    0|       0.0|168466|
|    1|       1.0| 85839|
|    0|       1.0| 31569|
+-----+----------+------+



In [ ]:
cm_pandas = confusion_matrix.toPandas()
cm_pandas.pivot(index = 'label', columns = 'prediction', values = 'count')

prediction,0.0,1.0
label,,
0,168466,31569
1,28055,85839


In [ ]:
tp = lr_predictions_test[(lr_predictions_test.label == 1) & (lr_predictions_test.prediction == 1)].count()
fp = lr_predictions_test[(lr_predictions_test.label == 0) & (lr_predictions_test.prediction == 1)].count()
fn = lr_predictions_test[(lr_predictions_test.label == 1) & (lr_predictions_test.prediction == 0)].count()
tn = lr_predictions_test[(lr_predictions_test.label == 0) & (lr_predictions_test.prediction == 0)].count()

---
# **Task 5 - Results Visualization or Printing (5 marks)**
---

In [ ]:
accuracy = (tp + tn) / (tp + fp + fn + tn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * (precision * recall) / (precision + recall)

print('accuracy: ', round(accuracy, 4) * 100)
print('precision: ', round(precision, 4) * 100)
print('recall: ', round(recall, 4) * 100)
print('f1 score: ', round(f1, 4) * 100)

accuracy:  81.01
precision:  73.11
recall:  75.37
f1 score:  74.22


---
# **Task 6 - LSEP Considerations (5 marks)**
---

# Computational efficacy

---

# **Task 7 - Convert ipynb to HTML for Turnitin submission [5 marks]**

---



In [ ]:
# install nbconvert (if facing the conversion error)
!pip3 install nbconvert

In [ ]:
%cd '/content/drive/MyDrive/Colab Notebooks'

In [ ]:
# convert ipynb to html and submit this HTML file
!jupyter nbconvert --to html Your_Group_ID_CRWK_CN7030.ipynb